In [1]:
# Run once if NLTK isn't installed
# !pip install nltk

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import string

# Download required NLTK resources the first time
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

print("Setup complete ✅")

Setup complete ✅


In [11]:
# Course database
courses = {
    1: "Python Programming for Students",
    2: "Machine Learning Using Python",
    3: "React Frontend Development",
    4: "Generative AI and Large Language Models",
    5: "Python Data Science and Machine Learning",
    6: "Full Stack Development Using React Node MongoDB",
    7: "Deep Learning with Python",
    8: "Natural Language Processing with Python"
}

In [13]:
# NLP Cleaning


def clean_text(text):
    """
    Full NLP pipeline:
    Tokenize → Lowercase → Remove punctuation → Remove stopwords → Lemmatize
    Returns a list of clean keywords.
    """
    # 1. Tokenize
    tokens = word_tokenize(text)

    # 2. Lowercase + remove punctuation
    tokens = [t.lower() for t in tokens if t.isalnum()]

    # 4. Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    # 5. Lemmatize with POS-aware tagging
    tagged = pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

    return lemmas


stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    """Map NLTK POS tag → WordNet POS tag for better lemmatization."""
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN  # default


# Quick test
print("Keywords:", clean_text("I want to learn Gen AI and Deep Learning"))

Keywords: ['want', 'learn', 'gen', 'ai', 'deep', 'learning']


In [14]:
# Pre-clean the course data
course_keywords = {
    cid: clean_text(title)
    for cid, title in courses.items()
}
course_keywords

{1: ['python', 'programming', 'student'],
 2: ['machine', 'learn', 'use', 'python'],
 3: ['react', 'frontend', 'development'],
 4: ['generative', 'ai', 'large', 'language', 'model'],
 5: ['python', 'data', 'science', 'machine', 'learning'],
 6: ['full', 'stack', 'development', 'use', 'react', 'node', 'mongodb'],
 7: ['deep', 'learning', 'python'],
 8: ['natural', 'language', 'processing', 'python']}

In [6]:
# Relevance scoring
def search_courses(query, exact_weight=2, partial_weight=1):
    """
    Search the course database for a query.

    Scoring:
      - exact keyword match in course  → +exact_weight (default 2)
      - partial/substring match        → +partial_weight (default 1)

    Returns a list of (course_id, title, score) sorted by score desc.
    """
    query_keywords = clean_text(query)

    if not query_keywords:
        return [], query_keywords

    results = []

    for cid, title in courses.items():
        title_keywords = course_keywords[cid]
        title_lower = title.lower()
        score = 0

        for kw in query_keywords:
            if kw in title_keywords:
                # Exact keyword match
                score += exact_weight
            elif kw in title_lower:
                # Partial/substring match
                score += partial_weight

        if score > 0:
            results.append((cid, title, score))

    # Sort highest score first
    results.sort(key=lambda x: x[2], reverse=True)
    return results, query_keywords

In [7]:
# Display the result
def display_results(query):
    results, keywords = search_courses(query)

    print("🔎 Search Keywords")
    print("-------------------")
    for kw in keywords:
        print(kw)

    print("\n📚 Search Results")
    print("-------------------")

    # Level 2 Challenge: no match message
    if not results:
        print("❌ No relevant courses found.")
        print("Try searching with different keywords.")
        return

    for rank, (cid, title, score) in enumerate(results, start=1):
        print(f"\n{rank}. {title}")
        print(f"   Score: {score}")

In [8]:
# Test with single query
display_results("I want to learn Python and Machine Learning")

🔎 Search Keywords
-------------------
want
learn
python
machine
learning

📚 Search Results
-------------------

1. Machine Learning Using Python
   Score: 7

2. Python Data Science and Machine Learning
   Score: 7

3. Deep Learning with Python
   Score: 5

4. Python Programming for Beginners
   Score: 2

5. Natural Language Processing with Python
   Score: 2


In [9]:
# Run all test queries
test_queries = [
    "python courses",
    "machine learning using python",
    "I want to learn React and Node",
    "generative artificial intelligence",
    "deep learning",
    "natural language processing",
    "full stack development",
]

for q in test_queries:
    print("=" * 50)
    print(f"QUERY: {q}")
    print("=" * 50)
    display_results(q)
    print()

QUERY: python courses
🔎 Search Keywords
-------------------
python
course

📚 Search Results
-------------------

1. Python Programming for Beginners
   Score: 2

2. Machine Learning Using Python
   Score: 2

3. Python Data Science and Machine Learning
   Score: 2

4. Deep Learning with Python
   Score: 2

5. Natural Language Processing with Python
   Score: 2

QUERY: machine learning using python
🔎 Search Keywords
-------------------
machine
learn
use
python

📚 Search Results
-------------------

1. Machine Learning Using Python
   Score: 8

2. Python Data Science and Machine Learning
   Score: 5

3. Deep Learning with Python
   Score: 3

4. Python Programming for Beginners
   Score: 2

5. Full Stack Development Using React Node MongoDB
   Score: 2

6. Natural Language Processing with Python
   Score: 2

QUERY: I want to learn React and Node
🔎 Search Keywords
-------------------
want
learn
react
node

📚 Search Results
-------------------

1. Full Stack Development Using React Node Mong

In [10]:
# Continuous search loop
print("🔎 Mini Search Engine — type 'exit' to quit\n")

while True:
    query = input("Enter search query: ").strip()

    if query.lower() == "exit":
        print("\n👋 Goodbye! Happy learning.")
        break

    if not query:
        print("⚠️  Please enter something.\n")
        continue

    print()
    display_results(query)
    print()

🔎 Mini Search Engine — type 'exit' to quit

Enter search query: nlp

🔎 Search Keywords
-------------------
nlp

📚 Search Results
-------------------
❌ No relevant courses found.
Try searching with different keywords.

Enter search query: deep learning

🔎 Search Keywords
-------------------
deep
learning

📚 Search Results
-------------------

1. Deep Learning with Python
   Score: 4

2. Python Data Science and Machine Learning
   Score: 2

3. Machine Learning Using Python
   Score: 1

Enter search query: Data Science

🔎 Search Keywords
-------------------
data
science

📚 Search Results
-------------------

1. Python Data Science and Machine Learning
   Score: 4

Enter search query: exit

👋 Goodbye! Happy learning.
